## Init

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import from_json, col, explode, arrays_zip, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, DoubleType

## Read from bronze table

In [0]:
df_bronze_read = spark.read.table("weather.raw_data")

## Define structures to unnest JSON data

In [0]:
hourly_schema = StructType([
    StructField("time", ArrayType(StringType())),
    StructField("temperature_2m", ArrayType(DoubleType())),
    StructField("relative_humidity_2m", ArrayType(DoubleType())),
    StructField("precipitation", ArrayType(DoubleType()))
])

payload_schema = StructType([
    StructField("hourly", hourly_schema)
])

## Unnest the raw payload

In [0]:
df_parsed = (
    df_bronze_read
    .withColumn(
        "parsed_json", 
        from_json(col("raw_payload"), payload_schema)
    )
)

# Create a new row for each weather measurement 
df_zipped = (
    df_parsed
    .withColumn(
        "zipped_data", 
        explode(arrays_zip(
            col("parsed_json.hourly.time"),
            col("parsed_json.hourly.temperature_2m"),
            col("parsed_json.hourly.relative_humidity_2m"),
            col("parsed_json.hourly.precipitation")
        ))
    )
)


## Select final columns

In [0]:

df_silver = (
    df_zipped
    .select(
        col("state_code"),
        to_timestamp(col("zipped_data.time")).alias("measurement_timestamp"),
        col("zipped_data.temperature_2m").alias("temperature_celsius"),
        col("zipped_data.relative_humidity_2m").alias("humidity_percentage"),
        col("zipped_data.precipitation").alias("precipitation_mm"),
        col("ingested_at")
    ).distinct()
)

## Write data in silver table
Using MERGE with (state_code + date of ingestion + hour of ingestion) as unique key 

In [ ]:
table_name = "weather.weather_hourly"

if not spark.catalog.tableExists(table_name):
    (
        df_silver
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
else: 
    target_table = DeltaTable.forName(spark, table_name)

    (
        target_table.alias("target")
        .merge(
            df_silver.alias("source"), 
            condition="""
            target.state_code = source.state_code AND 
            to_date(target.measurement_timestamp) = to_date(source.measurement_timestamp) AND 
            hour(target.measurement_timestamp) = hour(source.measurement_timestamp)
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

## Sanity check

In [ ]:
%sql
select * 
from weather.weather_hourly
order by state_code, measurement_timestamp desc
limit 10